## Setup

In [48]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

In [49]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

IMAGE_DIR = "./math/images/images/"
train = pd.read_csv("./math/train.csv")
test = pd.read_csv("./math/test.csv")

In [50]:
MODEL_LOCAL_PATH = "/home/ub089/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5"

In [51]:
train

,id,image_path,answer
0,0,images/0.jpg,20 ตารางเซนติเมตร
1,102,images/102.jpg,76
2,104,images/104.jpg,45
3,105,images/105.jpg,192
4,106,images/106.jpg,2
...,...,...,...
275,87,images/87.jpg,3749 จำนวน
276,91,images/91.jpg,125
277,95,images/95.jpg,2n+2
278,97,images/97.jpg,12


In [52]:
test

,id,image_path
0,1,images/1.jpg
1,10,images/10.jpg
2,100,images/100.jpg
3,101,images/101.jpg
4,103,images/103.jpg
...,...,...
415,92,images/92.jpg
416,93,images/93.jpg
417,94,images/94.jpg
418,96,images/96.jpg


In [53]:
test['answer'] = 0
test

,id,image_path,answer
0,1,images/1.jpg,0
1,10,images/10.jpg,0
2,100,images/100.jpg,0
3,101,images/101.jpg,0
4,103,images/103.jpg,0
...,...,...,...
415,92,images/92.jpg,0
416,93,images/93.jpg,0
417,94,images/94.jpg,0
418,96,images/96.jpg,0


## Inference

In [54]:
print(f"Loading Qwen2.5-VL-7B-Instruct directly from:\n{MODEL_LOCAL_PATH}")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_LOCAL_PATH,
    torch_dtype=torch.bfloat16, # Optimal for A100 GPUs
    device_map="auto", # Automatically splits across your 2x A100s
    local_files_only=True
)

Loading Qwen2.5-VL-7B-Instruct directly from:
/home/ub089/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [55]:
processor = AutoProcessor.from_pretrained(
    MODEL_LOCAL_PATH,
    local_files_only=True
)

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [56]:
system_prompt = (
    "You are a strict data extraction AI. Solve the math problem in the image "
    "and output ONLY the final answer. If the answer includes Thai units "
    "(such as ตารางเซนติเมตร or จำนวน), include them exactly as they should be written. "
    "Do not include any steps, reasoning, or conversational text. Just the final answer."
)

In [59]:
predicted_answers = []

print(f"Starting zero-shot inference on {len(test)} images...")
for index, row in tqdm(test.iterrows(), total=len(test)):
    # Construct exact image path based on your format: IMAGE_PATH + id + .jpg
    img_path = f"{IMAGE_DIR}{row['id']}.jpg"
    
    # Safety check: ensure image exists
    if not os.path.exists(img_path):
        print(f"Warning: Image not found at {img_path}. Skipping.")
        predicted_answers.append("Image Not Found")
        continue

    # Format the input for Qwen2.5-VL
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": system_prompt}
            ]
        }
    ]

    # Process text and vision inputs
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    # Generate the output
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=30, # Low token count because we only want the final answer
            temperature=0.0,   # Deterministic output for math accuracy
            do_sample=False
        )

    # Trim the input prompt tokens from the output
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    # Decode the prediction
    output_text = processor.batch_decode(
        generated_ids_trimmed, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )[0]

    predicted_answers.append(output_text.strip())

Starting zero-shot inference on 420 images...


100%|██████████| 420/420 [03:34<00:00,  1.96it/s]


In [63]:
test['answer'] = predicted_answers

In [65]:
submission = test.copy()
submission = submission.drop(columns=['image_path'])
submission.to_csv("submission.csv", index=False)